## 1. Install notebook dependencies

This cell installs the Python packages needed by this notebook:

- **databricks-mcp** — Databricks MCP integration libraries.
- **databricks-sdk** — lets Python interact with the Databricks workspace and Apps.
- **mcp** — the standard MCP Python SDK used to connect to our deployed MCP server.
- **databricks-openai** — OpenAI-compatible client used to call Databricks foundation models such as GPT-6 Luna.

We pin `mcp` below version 2 because the Databricks MCP client environment currently uses the MCP 1.x SDK.

After installing packages, `%restart_python` restarts the notebook Python process so the newly installed versions are loaded.

**Run this cell first when setting up a new notebook environment.**

In [0]:
%pip install --upgrade databricks-mcp databricks-sdk "mcp>=1.24,<2" "networkx>=3.0,<4"
%pip install -U databricks-openai
dbutils.library.restartPython()

## 2. Authenticate to the deployed MCP Server App

Our N2IA MCP server is deployed as a protected Databricks App. Before this notebook can call its `/mcp` endpoint, it needs an OAuth access token for that specific App.

This cell:

1. Finds the `mcp-n2ia-server-app` Databricks App.
2. Gets the App's OAuth client ID.
3. Gets the current notebook's Databricks credential.
4. Exchanges that credential for an OAuth token whose audience is our App.
5. Stores the resulting token in `audience_token`.

The token is later sent with MCP requests to authenticate this notebook to the Weather MCP server.

**Security:** Do not print or store the actual access token.

In [0]:
import requests
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# We already retrieved this before, but doing it here makes the cell self-contained.
app_client_id = w.apps.get("mcp-n2ia-server-app").oauth2_app_client_id

# Workspace URL, for example https://xxxx.cloud.databricks.com
workspace_url = w.config.host.rstrip("/")

# Get the notebook's internal token
notebook_token = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook()
    .getContext()
    .apiToken()
    .get()
)

# Exchange it for an OAuth token specifically for the Databricks App
response = requests.post(
    f"{workspace_url}/oidc/v1/token",
    data={
        "grant_type": "urn:ietf:params:oauth:grant-type:token-exchange",
        "subject_token": notebook_token,
        "subject_token_type": "urn:databricks:params:oauth:token-type:personal-access-token",
        "requested_token_type": "urn:ietf:params:oauth:token-type:access_token",
        "scope": "all-apis",
        "audience": app_client_id,
    },
)

response.raise_for_status()

audience_token = response.json()["access_token"]

print("OAuth token obtained successfully")
print("Token length:", len(audience_token))

## 3. Connect to the MCP Server and Discover Its Tools

Before involving the LLM, verify that this notebook can communicate directly with our deployed MCP server.

This cell:

1. Connects to the Databricks App's `/mcp` endpoint over HTTPS.
2. Sends the OAuth token obtained in the previous section.
3. Creates an MCP client session.
4. Performs the MCP initialization handshake.
5. Calls `list_tools()` to discover the tools exposed by the server.
6. Displays each tool's name and description.

At this stage there is **no LLM and no agent decision-making**. We are testing only:

**Notebook → MCP Client → MCP Server**

A successful result proves that authentication, networking, MCP communication, and tool discovery are working.

In [0]:
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

MCP_URL = "https://mcp-n2ia-server-app-7474654188777746.aws.databricksapps.com/mcp"

async with streamablehttp_client(
    MCP_URL,
    headers={"Authorization": f"Bearer {audience_token}"}
) as (read, write, _):

    async with ClientSession(read, write) as session:
        await session.initialize()

        tools = await session.list_tools()

        for tool in tools.tools:
            print(tool.name)

In [0]:
#Step 2 — Run the existing MCP test
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

MCP_URL = "https://mcp-n2ia-server-app-7474654188777746.aws.databricksapps.com/mcp"

async with streamablehttp_client(
    MCP_URL,
    headers={"Authorization": f"Bearer {audience_token}"}
) as (read, write, _):

    async with ClientSession(read, write) as session:
        await session.initialize()

        result = await session.call_tool(
            "analyze_contract_overlap",
            arguments={
                "start_date": "2024-10-01",
                "end_date": "2026-09-30",
                "agencies": [
                    "General Services Administration",
                    "Department of Homeland Security",
                    "Department of the Treasury"
                ],
                "keyword": "cybersecurity",
                "analysis_method": "both",
                "page_size": 25,
                "pages_per_agency": 1,
                "min_similarity": 0.3,
                "max_candidates": 5
            }
        )

        print("Tool error:", result.isError)

        for item in result.content:
            if item.type == "text":
                print(item.text[:10000])

In [0]:
# After rerunning your direct MCP tool call
analysis = data["analysis"]

print("Grouping method:", analysis.get("grouping_method"))
print("Function mismatches:",
      analysis.get("grouping_screened_out_function_mismatch"))

In [0]:
analysis = data["analysis"]

print("Groups:", analysis["group_count"])
print("Relationships retained:", analysis["grouping_relationship_count"])
print("Relationships filtered:", analysis["grouping_screened_out_relationships"])

for group in analysis["groups"]:
    print(f"\n{group['group_id']} — {group['service_functions']}")
    for contract in group["contracts"]:
        print(
            contract["award_id"],
            "|",
            contract["description"]
        )

In [0]:
analysis = data["analysis"]

print("Groups:", analysis["group_count"])
print("Relationships retained:", analysis["grouping_relationship_count"])
print("Function mismatches:", analysis["grouping_screened_out_function_mismatch"])

for group in analysis["groups"]:
    print(
        group["group_id"],
        group["service_functions"],
        [c["award_id"] for c in group["contracts"]]
    )

In [0]:
#Step 3 — Inspect the response
import json

if result.structuredContent:
    data = result.structuredContent
else:
    data = json.loads(result.content[0].text)

# Some MCP responses wrap the payload in "result".
if "analysis" not in data and isinstance(data.get("result"), dict):
    data = data["result"]

analysis = data["analysis"]

print("Retrieved contracts:", data["retrieved_count"])
print("Analysis method:", data["analysis_method"])
print("Ranked candidates:", len(analysis.get("ranked_candidates", [])))
print("Total matches:", analysis.get("ranked_candidate_count"))
print("Counts:", analysis.get("counts"))

In [0]:
for key, value in analysis.items():
    if "group" in key or "function" in key:
        if key != "groups":
            print(key, ":", value)

In [0]:
analysis = data["analysis"]

print("Groups:", analysis["group_count"])
print("Relationships retained:", analysis["grouping_relationship_count"])
print("Function mismatches:", analysis["grouping_screened_out_function_mismatch"])

for group in analysis["groups"]:
    print(
        group["group_id"],
        group["service_functions"],
        [c["award_id"] for c in group["contracts"]]
    )

In [0]:
#Step 4 — Display the top 10 as a table
import pandas as pd

candidates = analysis.get("ranked_candidates", [])

df = pd.json_normalize(candidates, sep="_")

display(df)

In [0]:
#Step 5 — Inspect the highest-ranked candidate
if candidates:
    print(json.dumps(candidates[0], indent=2, default=str))
else:
    print("No ranked candidates returned.")

In [0]:
analysis = data["analysis"]

print("Total groups:", analysis["group_count"])
print("Relationships retained:", analysis["grouping_relationship_count"])
print("Relationships filtered:", analysis["grouping_screened_out_relationships"])

for group in analysis["groups"]:
    print(
        group["group_id"],
        "Contracts:", group["contract_count"],
        "Agencies:", group["agency_count"],
        "Relationships:", group["relationship_count"],
        "Density:", group["connection_density"]
    )

In [0]:
analysis = data["analysis"]

print("Groups:", analysis["group_count"])
print("Relationships retained:", analysis["grouping_relationship_count"])
print("Function mismatches:", analysis["grouping_screened_out_function_mismatch"])

for group in analysis["groups"]:
    print(
        group["group_id"],
        group["service_functions"],
        [c["award_id"] for c in group["contracts"]]
    )

In [0]:
for group in analysis["groups"]:
    print(f"\n{group['group_id']} — {group['contract_count']} contracts")

    for c in group["contracts"]:
        print(f"  {c['award_id']} | {c['awarding_agency']}")
        print(f"  {c['description']}")
        print(f"  PSC: {c['psc']} | NAICS: {c['naics']}")
        print()

In [0]:
analysis = data["analysis"]

print("Total groups:", analysis["group_count"])

import pandas as pd

rows = []

for group in analysis["groups"]:
    for contract in group["contracts"]:
        rows.append({
            "Group": group["group_id"],
            "Contract": contract["award_id"],
            "Agency": contract["awarding_agency"],
            "Contractor": contract["contractor"],
            "Description": contract["description"],
            "PSC": contract["psc"],
            "NAICS": contract["naics"]
        })

display(pd.DataFrame(rows))

In [0]:
import pandas as pd

rows = []

for i, c in enumerate(analysis["ranked_candidates"], 1):
    rows.append({
        "Rank": i,
        "Contract A": c["left"]["award_id"],
        "Agency A": c["left"]["awarding_agency"],
        "Contract B": c["right"]["award_id"],
        "Agency B": c["right"]["awarding_agency"],
        "PSC/NAICS": c["classification_alignment"],
        "Lexical": c["lexical_similarity"],
        "Semantic": c["semantic_similarity"],
        "Overlap Days": c["overlap_days"],
        "Method": c["match_method"]
    })

display(pd.DataFrame(rows))

In [0]:
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

# MCP endpoint exposed by our deployed Databricks App.
server_url = (
    "https://mcp-n2ia-server-app-7474654188777746.aws.databricksapps.com/mcp"
)


async def test_mcp_connection():
    headers = {
        "Authorization": f"Bearer {audience_token}"
    }

    # Connect to the MCP server over HTTPS.
    async with streamablehttp_client(
        server_url,
        headers=headers
    ) as (read_stream, write_stream, get_session_id):

        # Create an MCP client session.
        async with ClientSession(
            read_stream,
            write_stream
        ) as session:

            # Perform the MCP initialization handshake.
            await session.initialize()

            # Ask the MCP server which tools it exposes.
            result = await session.list_tools()

            print("Connected successfully.")
            print("Number of tools:", len(result.tools))

            for tool in result.tools:
                print("\nTOOL:", tool.name)
                print("DESCRIPTION:", tool.description)


await test_mcp_connection()

In [0]:
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client


async def get_mcp_tools():
    headers = {
        "Authorization": f"Bearer {audience_token}"
    }

    async with streamablehttp_client(
        server_url,
        headers=headers
    ) as (read_stream, write_stream, get_session_id):

        async with ClientSession(
            read_stream,
            write_stream
        ) as session:

            await session.initialize()

            result = await session.list_tools()

            return result.tools


mcp_tools = await get_mcp_tools()

for tool in mcp_tools:
    print("\nNAME:", tool.name)
    print("DESCRIPTION:", tool.description)
    print("SCHEMA:", tool.inputSchema)

   
## 4. Define USAspending.gov API Tools

This cell creates a local Python toolkit for querying the **USAspending.gov** public API — the official source of federal spending data.

It defines five tool functions:

| Tool | Description |
|------|-------------|
| `search_spending_by_award` | Search contract and grant awards by keyword, date range, agency, and award type. |
| `get_agency_overview` | Get a high-level spending summary for a federal agency by fiscal year. |
| `get_award_details` | Retrieve full details of a specific award by its unique ID. |
| `search_spending_by_category` | Aggregate spending by category (agency, recipient, NAICS, state, etc.). |
| `spending_over_time` | Get spending trends aggregated by fiscal year, quarter, or month. |

The cell also:

- Builds an **OpenAI-compatible tool definitions list** (`usaspending_tools`) so an LLM can invoke these tools via function calling.
- Provides a **dispatcher function** (`dispatch_usaspending_tool`) for use inside an agent loop.
- Runs a **smoke test** searching for recent "cloud computing" contract awards to verify the API is reachable.

These tools operate independently of the MCP server — they call the USAspending.gov REST API directly from the notebook.

In [0]:
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client
import requests
import json

# ---------------------------------------------------------------------------
# USAspending.gov API tool functions
# Docs: https://api.usaspending.gov/docs/endpoints
# ---------------------------------------------------------------------------

USA_SPENDING_BASE = "https://api.usaspending.gov/api/v2"


def search_spending_by_award(filters: dict, limit: int = 10, page: int = 1) -> dict:
    """
    Search award spending using the USAspending Awards API.
    Endpoint: POST /api/v2/search/spending_by_award/

    Parameters
    ----------
    filters : dict
        Filter object per USAspending API spec. Common keys:
        - "keywords": list[str]          – free-text keyword search
        - "time_period": list[dict]      – [{"start_date": "YYYY-MM-DD", "end_date": "YYYY-MM-DD"}]
        - "award_type_codes": list[str]  – e.g. ["A","B","C","D"] for contracts
        - "agencies": list[dict]         – agency filter objects
    limit : int   – Results per page (max 100).
    page  : int   – Page number (1-indexed).
    """
    url = f"{USA_SPENDING_BASE}/search/spending_by_award/"
    payload = {
        "filters": filters,
        "limit": limit,
        "page": page,
        "fields": [
            "Award ID", "Recipient Name", "Award Amount",
            "Awarding Agency", "Start Date", "End Date",
            "Description",
        ],
        "sort": "Award Amount",
        "order": "desc",
        "subawards": False,
    }
    resp = requests.post(url, json=payload, timeout=30)
    resp.raise_for_status()
    return resp.json()


def get_agency_overview(toptier_agency_code: str, fiscal_year: int = 2024) -> dict:
    """
    High-level overview of a federal agency's spending.
    Endpoint: GET /api/v2/agency/{toptier_code}/
    """
    url = f"{USA_SPENDING_BASE}/agency/{toptier_agency_code}/"
    resp = requests.get(url, params={"fiscal_year": fiscal_year}, timeout=30)
    resp.raise_for_status()
    return resp.json()


def get_award_details(award_id: str) -> dict:
    """
    Detailed information about a specific award.
    Endpoint: GET /api/v2/awards/{award_id}/
    """
    url = f"{USA_SPENDING_BASE}/awards/{award_id}/"
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    return resp.json()


def search_spending_by_category(category: str, filters: dict, limit: int = 10, page: int = 1) -> dict:
    """
    Aggregate spending by category (awarding_agency, recipient, cfda, naics, etc.).
    Endpoint: POST /api/v2/search/spending_by_category/{category}/
    """
    url = f"{USA_SPENDING_BASE}/search/spending_by_category/{category}/"
    payload = {"filters": filters, "limit": limit, "page": page}
    resp = requests.post(url, json=payload, timeout=30)
    resp.raise_for_status()
    return resp.json()


def spending_over_time(filters: dict, group: str = "fiscal_year") -> dict:
    """
    Spending aggregated over time.
    Endpoint: POST /api/v2/search/spending_over_time/
    group: "fiscal_year", "quarter", or "month".
    """
    url = f"{USA_SPENDING_BASE}/search/spending_over_time/"
    payload = {"filters": filters, "group": group, "subawards": False}
    resp = requests.post(url, json=payload, timeout=30)
    resp.raise_for_status()
    return resp.json()


# ---------------------------------------------------------------------------
# OpenAI-function-calling-compatible tool definitions
# Pass this list to your LLM so it can invoke these tools by name.
# ---------------------------------------------------------------------------
usaspending_tools = [
    {
        "type": "function",
        "function": {
            "name": "search_spending_by_award",
            "description": "Search federal contract and grant awards on USAspending.gov by keyword, date, agency, and award type.",
            "parameters": {
                "type": "object",
                "properties": {
                    "filters": {
                        "type": "object",
                        "description": "USAspending filter object. Keys: keywords (list[str]), time_period (list of {start_date, end_date}), award_type_codes (list[str]: A-D=contracts, 02-10=grants), agencies (list[dict]).",
                    },
                    "limit": {"type": "integer", "description": "Results per page (max 100).", "default": 10},
                    "page":  {"type": "integer", "description": "Page number.", "default": 1},
                },
                "required": ["filters"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_agency_overview",
            "description": "Get a high-level spending overview of a federal agency by its top-tier agency code and fiscal year.",
            "parameters": {
                "type": "object",
                "properties": {
                    "toptier_agency_code": {"type": "string", "description": "Top-tier agency code, e.g. '012' for Agriculture, '097' for Defense."},
                    "fiscal_year": {"type": "integer", "description": "Federal fiscal year.", "default": 2024},
                },
                "required": ["toptier_agency_code"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_award_details",
            "description": "Retrieve full details of a specific federal award by its generated unique award ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "award_id": {"type": "string", "description": "The generated unique award ID."},
                },
                "required": ["award_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search_spending_by_category",
            "description": "Aggregate federal spending by a category such as awarding_agency, recipient, cfda, naics, psc, state_territory, county, or federal_account.",
            "parameters": {
                "type": "object",
                "properties": {
                    "category": {"type": "string", "description": "Category to group by (awarding_agency, recipient, cfda, naics, psc, state_territory, etc.)."},
                    "filters":  {"type": "object", "description": "USAspending filter object."},
                    "limit":    {"type": "integer", "default": 10},
                    "page":     {"type": "integer", "default": 1},
                },
                "required": ["category", "filters"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "spending_over_time",
            "description": "Get federal spending aggregated over time (by fiscal_year, quarter, or month).",
            "parameters": {
                "type": "object",
                "properties": {
                    "filters": {"type": "object", "description": "USAspending filter object."},
                    "group":   {"type": "string", "enum": ["fiscal_year", "quarter", "month"], "default": "fiscal_year"},
                },
                "required": ["filters"],
            },
        },
    },
]

# ---------------------------------------------------------------------------
# Dispatcher – call a tool by name (used inside an agent loop)
# ---------------------------------------------------------------------------
def dispatch_usaspending_tool(tool_name: str, arguments: dict):
    """Execute a USAspending tool by name and return the JSON result."""
    tool_map = {
        "search_spending_by_award": search_spending_by_award,
        "get_agency_overview": get_agency_overview,
        "get_award_details": get_award_details,
        "search_spending_by_category": search_spending_by_category,
        "spending_over_time": spending_over_time,
    }
    fn = tool_map.get(tool_name)
    if fn is None:
        raise ValueError(f"Unknown tool: {tool_name}")
    return fn(**arguments)


# ---------------------------------------------------------------------------
# Quick smoke test – search for recent "cloud computing" contract awards
# ---------------------------------------------------------------------------
result = search_spending_by_award(
    filters={
        "keywords": ["cloud computing"],
        "time_period": [{"start_date": "2024-01-01", "end_date": "2024-12-31"}],
        "award_type_codes": ["A", "B", "C", "D"],
    },
    limit=5,
)

print(f"Total results: {result.get('page_metadata', {}).get('total', 'N/A')}")
print(f"Showing {len(result.get('results', []))} awards:\n")
for award in result.get("results", []):
    print(f"  Award ID   : {award.get('Award ID')}")
    print(f"  Recipient  : {award.get('Recipient Name')}")
    print(f"  Amount     : ${award.get('Award Amount', 0):,.2f}")
    print(f"  Agency     : {award.get('Awarding Agency')}")
    print(f"  Description: {(award.get('Description') or '')[:100]}")
    print()

print("\n✅ USAspending tools ready. 'usaspending_tools' list can be passed to an LLM.")
print(f"   {len(usaspending_tools)} tools defined: {[t['function']['name'] for t in usaspending_tools]}")

   
## 5. Define GovInfo API Tools

This cell creates a local Python toolkit for querying the **GovInfo API** — the official repository for U.S. Government publications.

It defines a `search_govinfo` function that:

1. Retrieves the **GovInfo API key** from environment variables (loaded via `.env`).
2. Searches the GovInfo API by keyword and collection (e.g., `BUDGET`).
3. Returns results sorted by relevance score.

A **smoke test** searches for documents related to "cybersecurity" in the BUDGET collection to verify the API is reachable and the API key is valid.

> **Note:** This cell loads the API key from a local `.env` file. Later cells transition to using **Databricks Secrets** for secure credential management.

In [0]:
import os
import requests
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()

API_KEY = os.getenv("GOVINFO_API_KEY")

if not API_KEY:
    raise ValueError("GOVINFO_API_KEY not found in .env")

def search_govinfo(query, collection="BUDGET", limit=10):
    url = "https://api.govinfo.gov/search"

    payload = {
        "query": f"collection:({collection}) AND ({query})",
        "pageSize": str(limit),
        "offsetMark": "*",
        "sorts": [
            {"field": "score", "sortOrder": "DESC"}
        ]
    }

    response = requests.post(
        url,
        params={"api_key": API_KEY},
        json=payload,
        timeout=30
    )

    response.raise_for_status()
    return response.json()


# Test
results = search_govinfo('"cybersecurity"', limit=5)

for item in results.get("results", []):
    print(item.get("title"))
    print(item.get("packageId"))
    print(item.get("dateIssued"))
    print("---")

   
## 6. Scrape GAO Duplication & Cost Savings Data

This cell scrapes the **U.S. Government Accountability Office (GAO)** [Duplication & Cost Savings](https://www.gao.gov/duplication-cost-savings/duplication-cost-savings-recommendations) page to find and download the associated CSV dataset.

It:

1. Fetches the GAO recommendations page using `requests` and parses it with `BeautifulSoup`.
2. Extracts all CSV download links from the page.
3. Downloads the first CSV file found and loads it into a **pandas DataFrame**.
4. Displays the shape, column names, and a preview of the data.

This dataset contains GAO's recommendations for reducing fragmentation, overlap, and duplication across federal programs — useful for identifying cost-saving opportunities in government spending.

In [0]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

page_url = (
    "https://www.gao.gov/duplication-cost-savings/"
    "duplication-cost-savings-recommendations"
)

headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'}
response = requests.get(page_url, headers=headers, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

csv_links = [
    urljoin(page_url, a["href"])
    for a in soup.find_all("a", href=True)
    if "csv" in a["href"].lower()
    or a.get_text(strip=True).lower() == "csv"
]

print("CSV links:", csv_links)

if csv_links:
    df = pd.read_csv(csv_links[0])
    print(df.shape)
    print(df.columns.tolist())
    display(df.head())

   
   
## 7. Query SAM.gov Contract Awards API

This cell tests connectivity to the **SAM.gov Contract Awards API** — the official source for federal procurement data managed by the General Services Administration (GSA).

It:

1. Loads the **SAM.gov API key** from environment variables (via `.env`).
2. Queries the Contract Awards endpoint for **FY2025** awards from **GSA** (department code `4700`).
3. Returns a small sample (`limit=2`) to verify the API is reachable and the key is valid.

> **Note:** This cell uses a local `.env` file for the API key. A later section transitions to **Databricks Secrets** for secure credential management.

In [0]:
import os
import requests
from dotenv import load_dotenv

load_dotenv(override=True)

api_key = os.getenv("SAM_API_KEY")

url = (
    "https://api.sam.gov/contract-awards/v1/search"
    f"?api_key={api_key}"
    "&fiscalYear=2025"
    "&contractingDepartmentCode=4700"
    "&limit=2"
)

response = requests.get(url, timeout=30)
response.raise_for_status()

data = response.json()
print(data)

## Creating Databricks Secrets

I ran these commands to create a secret scope and secret keys for accessing SAM and GovInfo (using my gsa.gov email address).

### Setup Commands

| Command | Description |
|---------|-------------|
| `databricks secrets create-scope n2ia-federal-intelligence` | Created the **scope** — a named container for storing related secrets, such as API keys and passwords. |
| `databricks secrets put-secret n2ia-federal-intelligence sam-api-key` | Created the secret for accessing **SAM**. You will be prompted to enter the SAM API Key. |
| `databricks secrets put-secret n2ia-federal-intelligence govinfo-api-key` | Created the secret for accessing **GovInfo**. You will be prompted to enter the GovInfo API Key. |

### Inspection Commands

| Command | Description |
|---------|-------------|
| `databricks secrets list-scopes` | List all available scopes. |
| `databricks secrets list-acls n2ia-federal-intelligence` | List the permissions associated with the scope. |
| `databricks secrets list-secrets n2ia-federal-intelligence` | List the secrets stored within the scope. |

   
## 8. Query SAM.gov Using Databricks Secrets

This cell repeats the SAM.gov Contract Awards API call from earlier, but now retrieves the API key securely using **Databricks Secrets** instead of a local `.env` file.

It:

1. Retrieves the **SAM.gov API key** from the `n2ia-federal-intelligence` secret scope using `dbutils.secrets.get()`.
2. Queries the SAM.gov Contract Awards endpoint for **FY2025** awards from **GSA** (department code `4700`).
3. Returns a small sample (`limit=2`) and prints the result.

This demonstrates the transition from local environment variables to Databricks' secure secret management for production-ready credential handling.

In [0]:
import requests

# Retrieve SAM.gov API key from Databricks Secrets
api_key = dbutils.secrets.get(
    scope="n2ia-federal-intelligence",
    key="sam-api-key"
)

# SAM.gov Contract Awards API
url = "https://api.sam.gov/contract-awards/v1/search"

params = {
    "api_key": api_key,
    "fiscalYear": "2025",
    "contractingDepartmentCode": "4700",  # GSA
    "limit": 2
}

# Call the API
response = requests.get(url, params=params, timeout=30)

print("HTTP Status:", response.status_code)

if response.ok:
    data = response.json()
    print("API call successful!")
    print(data)
else:
    print("Error:", response.text[:1000])

   
   
## 9. Exchange Notebook Token for OAuth Access Token

This cell obtains a scoped **OAuth access token** for authenticating with the deployed Databricks App (`n2ia-mcpserver-test`).

It:

1. Retrieves the **OAuth app client ID** from the deployed Databricks App using the Databricks SDK.
2. Extracts the **notebook's internal personal access token** via `dbutils`.
3. Performs an **OAuth token exchange** by posting to the workspace's OIDC endpoint (`/oidc/v1/token`), trading the notebook token for a scoped access token with `all-apis` scope.
4. Stores the resulting `audience_token` for use in subsequent cells that call the MCP server.

This token exchange pattern allows notebook code to securely authenticate against Databricks Apps without embedding long-lived credentials.

In [0]:
import requests
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# We already retrieved this before, but doing it here makes the cell self-contained.
app_client_id = w.apps.get("n2ia-mcpserver-test").oauth2_app_client_id

# Workspace URL, for example https://xxxx.cloud.databricks.com
workspace_url = w.config.host.rstrip("/")

# Get the notebook's internal token
notebook_token = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook()
    .getContext()
    .apiToken()
    .get()
)

# Exchange it for an OAuth token specifically for the Databricks App
response = requests.post(
    f"{workspace_url}/oidc/v1/token",
    data={
        "grant_type": "urn:ietf:params:oauth:grant-type:token-exchange",
        "subject_token": notebook_token,
        "subject_token_type": "urn:databricks:params:oauth:token-type:personal-access-token",
        "requested_token_type": "urn:ietf:params:oauth:token-type:access_token",
        "scope": "all-apis",
        "audience": app_client_id,
    },
)

response.raise_for_status()

audience_token = response.json()["access_token"]

print("OAuth token obtained successfully")
print("Token length:", len(audience_token))

   
   
## 10. Test MCP Server Connection

This cell verifies end-to-end connectivity to the deployed **MCP (Model Context Protocol) server** running as a Databricks App.

It:

1. Defines the MCP server endpoint URL for the `n2ia-mcpserver-test` Databricks App.
2. Connects to the `/mcp` endpoint using the `streamablehttp_client`, authenticating with the **OAuth access token** obtained in the previous cell.
3. Performs the MCP **initialization handshake** to establish a client session.
4. Calls `list_tools()` to discover all tools exposed by the MCP server.
5. Prints the **name** and **description** of each available tool to confirm the server is reachable and responding correctly.

In [0]:
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

# MCP endpoint exposed by our deployed Databricks App.
server_url = (
    "https://n2ia-mcpserver-test-7474654188777746.aws.databricksapps.com/mcp"
)


async def test_mcp_connection():
    headers = {
        "Authorization": f"Bearer {audience_token}"
    }

    # Connect to the MCP server over HTTPS.
    async with streamablehttp_client(
        server_url,
        headers=headers
    ) as (read_stream, write_stream, get_session_id):

        # Create an MCP client session.
        async with ClientSession(
            read_stream,
            write_stream
        ) as session:

            # Perform the MCP initialization handshake.
            await session.initialize()

            # Ask the MCP server which tools it exposes.
            result = await session.list_tools()

            print("Connected successfully.")
            print("Number of tools:", len(result.tools))

            for tool in result.tools:
                print("\nTOOL:", tool.name)
                print("DESCRIPTION:", tool.description)


await test_mcp_connection()

   
   
## 9. Test MCP Server Connection via OAuth

This cell exchanges the notebook's internal token for an **OAuth access token** scoped to the deployed Databricks App, then uses it to connect to the **MCP (Model Context Protocol) server** over HTTPS.

It:

1. Retrieves the **OAuth app client ID** from the deployed Databricks App (`n2ia-mcpserver-test`).
2. Exchanges the notebook's personal access token for a scoped **OAuth token** via the workspace OIDC endpoint.
3. Connects to the MCP server's `/mcp` endpoint using the `streamablehttp_client`.
4. Performs the MCP **initialization handshake** and lists all available tools exposed by the server.
5. Prints the name and description of each tool to verify the connection is working.

In [0]:
async def test_sam_api():
    headers = {
        "Authorization": f"Bearer {audience_token}"
    }

    async with streamablehttp_client(
        server_url,
        headers=headers
    ) as (read_stream, write_stream, get_session_id):

        async with ClientSession(
            read_stream,
            write_stream
        ) as session:

            await session.initialize()

            result = await session.call_tool(
                "test_sam_api",
                arguments={}
            )

            print("SAM.gov API Response:")

            for item in result.content:
                if item.type == "text":
                    print(item.text)


await test_sam_api()

In [0]:
%pip install sentence-transformers

In [0]:
from sentence_transformers import SentenceTransformer, util

# Downloads the model on first use, then caches it
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

contracts = [
    "Enterprise cloud infrastructure operations and support",
    "Cloud hosting, engineering, and infrastructure management",
    "Cybersecurity awareness training services"
]

embeddings = model.encode(contracts, convert_to_tensor=True)

similarities = util.cos_sim(embeddings, embeddings)

print(similarities)

In [0]:
%sql
-- Run in Databricks SQL Editor or a notebook %sql cell.
-- Schema name contains a dot, so it must be backtick-quoted.
-- If your actual schema uses an underscore, replace `satish.venkataraman` below.

CREATE TABLE IF NOT EXISTS users.satish_venkataraman.cache_contract_awards (
  source_award_key STRING NOT NULL COMMENT 'USAspending generated internal award identifier; logical upsert key',
  award_id STRING,
  awarding_agency STRING,
  contractor STRING,
  recipient_uei STRING,
  description STRING,
  psc STRING,
  naics STRING,
  performance_start_date STRING,
  performance_end_date STRING,
  award_obligations_usd DOUBLE,
  last_modified_date STRING,
  source_url STRING,
  retrieved_at_utc TIMESTAMP
)
USING DELTA
COMMENT 'Latest retrieved contract details; upsert by source_award_key in application logic';

CREATE TABLE IF NOT EXISTS users.satish_venkataraman.cache_contract_searches (
  search_id STRING NOT NULL COMMENT 'Unique identifier for this immutable search snapshot',
  search_key STRING NOT NULL COMMENT 'Hash of normalized filters and retrieval settings',
  search_parameters STRING COMMENT 'JSON of search parameters including dates, agencies, keyword and pagination',
  retrieved_at_utc TIMESTAMP,
  expires_at_utc TIMESTAMP,
  contract_keys ARRAY<STRING> COMMENT 'Source award keys in returned result order',
  contract_snapshot_json STRING COMMENT 'JSON array of exact award records returned for reproducible evaluation',
  contract_count INT,
  coverage STRING COMMENT 'JSON array of per-agency pagination and error details',
  complete BOOLEAN COMMENT 'Whether all matching result pages were exhausted',
  limitations STRING
)
USING DELTA
COMMENT 'Search result snapshots; retain historical rows for evaluation reproducibility';

CREATE TABLE IF NOT EXISTS users.satish_venkataraman.cache_contract_embeddings (
  description_hash STRING NOT NULL COMMENT 'Hash of exact description text used for embedding',
  model_name STRING NOT NULL,
  model_version STRING NOT NULL COMMENT 'Model revision or local artifact version identifier',
  embedding ARRAY<FLOAT> COMMENT 'Normalized embedding vector; 384 dimensions for all-MiniLM-L6-v2',
  created_at_utc TIMESTAMP
)
USING DELTA
COMMENT 'Reusable description embeddings; logical key is description_hash + model_name + model_version';

-- Verify the tables were created.
SHOW TABLES IN users.satish_venkataraman LIKE 'cache_contract_*';

In [0]:
# Testing the trace
import mlflow
import json

mlflow.set_tracking_uri("databricks")

trace = mlflow.get_trace(
    "tr-7f4fdc392f5ec80e9731aa9a5ae96dd7"
)

spans = trace.search_spans(name="analyze_contract_overlap")

for span in spans:
    output = span.outputs
    raw = output if isinstance(output, str) else json.dumps(output, default=str)

    print("Output length:", len(raw))
    print("Contains truncation marker:", "[truncated]" in raw)
    print("Last 300 characters:", raw[-300:])